# Ajnyana — Tokenizer

Train BPE tokenizer dari Ajnyana corpus.

- **Algo:** Byte-Level BPE (GPT-2 style)
- **Vocab:** 16,000
- **Special token:** `<|endoftext|>`
- **Input:** `data/processed/train.txt`
- **Output:** `tokenizer/vocab.json` + `tokenizer/merges.txt`

In [1]:
from tokenizers import ByteLevelBPETokenizer
import os
import json
import random

os.makedirs('../tokenizer', exist_ok=True)

TRAIN_TXT = '../data/processed/train.txt'
VAL_TXT   = '../data/processed/val.txt'
VOCAB_SIZE = 16_000

print('Libraries OK')
print(f'Train: {os.path.getsize(TRAIN_TXT)/1e6:.1f} MB')

Libraries OK
Train: 412.9 MB


## 1. Train BPE Tokenizer

In [2]:
print('Training BPE tokenizer...')

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[TRAIN_TXT],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=['<|endoftext|>']
)

print(f'Done!')
print(f'Vocab size: {tokenizer.get_vocab_size():,}')

Training BPE tokenizer...



Done!
Vocab size: 16,000


## 2. Save

In [3]:
tokenizer.save_model('../tokenizer')

# Save config
config = {
    'vocab_size': tokenizer.get_vocab_size(),
    'model_type': 'byte-level-bpe',
    'special_tokens': {'endoftext': '<|endoftext|>'},
    'trained_on': 'ajnyana-corpus-v1',
    'date': '2026-06-25'
}
with open('../tokenizer/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved:')
for f in os.listdir('../tokenizer'):
    size = os.path.getsize(f'../tokenizer/{f}')
    print(f'  tokenizer/{f} ({size/1e3:.1f} KB)')

Saved:
  tokenizer/config.json (0.2 KB)
  tokenizer/merges.txt (124.1 KB)
  tokenizer/vocab.json (226.0 KB)


## 3. Test — Encode / Decode

In [4]:
test_texts = [
    'Bahasa Sunda mangrupa basa nu digunakeun ku urang Sunda.',
    'Gunung Gede mangrupa salah sahiji gunung nu aya di Jawa Kulon.',
    'Urang Sunda téh boga budaya nu beunghar pisan.',
]

EOT_ID = tokenizer.token_to_id('<|endoftext|>')
print(f'<|endoftext|> id: {EOT_ID}\n')

for text in test_texts:
    enc = tokenizer.encode(text)
    dec = tokenizer.decode(enc.ids)
    print(f'Text:    {text}')
    print(f'Tokens:  {enc.tokens}')
    print(f'N tokens: {len(enc.ids)}')
    print(f'Decoded: {dec}')
    print(f'Lossless: {text == dec}')
    print()

<|endoftext|> id: 0

Text:    Bahasa Sunda mangrupa basa nu digunakeun ku urang Sunda.
Tokens:  ['Bahasa', 'ĠSunda', 'Ġmangrupa', 'Ġbasa', 'Ġnu', 'Ġdigunakeun', 'Ġku', 'Ġurang', 'ĠSunda', '.']
N tokens: 10
Decoded: Bahasa Sunda mangrupa basa nu digunakeun ku urang Sunda.
Lossless: True

Text:    Gunung Gede mangrupa salah sahiji gunung nu aya di Jawa Kulon.
Tokens:  ['Gunung', 'ĠGed', 'e', 'Ġmangrupa', 'Ġsalah', 'Ġsahiji', 'Ġgunung', 'Ġnu', 'Ġaya', 'Ġdi', 'ĠJawa', 'ĠKulon', '.']
N tokens: 13
Decoded: Gunung Gede mangrupa salah sahiji gunung nu aya di Jawa Kulon.
Lossless: True

Text:    Urang Sunda téh boga budaya nu beunghar pisan.
Tokens:  ['Urang', 'ĠSunda', 'ĠtÃ©h', 'Ġboga', 'Ġbudaya', 'Ġnu', 'Ġbeunghar', 'Ġpisan', '.']
N tokens: 9
Decoded: Urang Sunda téh boga budaya nu beunghar pisan.
Lossless: True



## 4. Coverage Check

In [5]:
with open(VAL_TXT, 'r', encoding='utf-8') as f:
    val_docs = [d.strip() for d in f.read().split('\n\n') if d.strip()]

sample = random.sample(val_docs, min(1000, len(val_docs)))

total_tokens = 0
total_words  = 0
for doc in sample:
    enc = tokenizer.encode(doc)
    total_tokens += len(enc.ids)
    total_words  += len(doc.split())

ratio = total_tokens / total_words
est_total_tokens = int(61_189_263 * ratio)

print(f'Val sample:        {len(sample):,} docs')
print(f'Words:             {total_words:,}')
print(f'Tokens:            {total_tokens:,}')
print(f'Tokens/word ratio: {ratio:.2f}')
print(f'Est. total corpus: {est_total_tokens:,} tokens')
print()
# Fer nanoGPT ~10M params: idealnya 200M+ tokens, tapi 100M cukup buat poc
print(f'Target 200M? {"✅" if est_total_tokens >= 200_000_000 else "⚠️ under 200M — OK for small model"}')

Val sample:        1,000 docs
Words:             149,016
Tokens:            297,801
Tokens/word ratio: 2.00
Est. total corpus: 122,283,672 tokens

Target 200M? ⚠️ under 200M — OK for small model


## Next

Tokenizer trained & saved → `03_architecture.ipynb` — nanoGPT model definition + parameter count.